# Lo scenario e i dati

**Azienda:** ToysGroup distribuisce giocattoli tramite una rete di rivenditori in diverse aree geografiche. I dati del Modulo 3 sono esportati in quattro file di formato diverso, forniti dal docente, da leggere, pulire, analizzare e modellare in Python.

### Anagrafiche · CSV ed Excel
- `products.csv`: ProductID, ProductName, Category, ListPrice.
- `regions.xlsx`: RegionID, State, SalesRegion (lo stato appartiene a una regione di vendita).

**Chiavi: ProductID e RegionID.**

### Vendite · JSON
- `sales.json`: SalesID, ProductID, RegionID, ResellerID, SalesDate, Quantity, SalesAmount.
- Una riga per transazione, con valori mancanti in Quantity e SalesAmount.

**Fatti di vendita collegati alle anagrafiche.**

### Rivenditori · CSV
- `resellers.csv`: ResellerID, RegionID, Ordini12m, GiorniUltimoOrdine, ScontoMedio, Reclami.
- Churn = 1 se il rivenditore non riordina da oltre 6 mesi.

**Base per classificazione e clustering.**

---

> **Consegna finale (entro 2 giornate lavorative):** un unico notebook .ipynb eseguibile dall'inizio senza errori, con le celle nell'ordine dei task; ogni cella di codice è preceduta da una cella Markdown con il numero del task e la risposta richiesta. Notebook e dati vanno su GitHub (Task 7).

## Costrutti disponibili

| Concetto | Definizione | Esempio |
|---|---|---|
| List comprehension | costruisce una nuova lista applicando un'espressione a ogni elemento di un iterabile, con un filtro if opzionale | `[p["ListPrice"] for p in products if p["Category"] == "Bikes"]` |
| `.loc[]` e `.iloc[]` | `.loc[]` seleziona righe e colonne per etichetta o maschera booleana; `.iloc[]` per posizione intera | `df.loc[df["ListPrice"] > 30, "ProductName"]`; `df.iloc[10:20, :3]` |
| `groupby()` e `agg()` | divide il DataFrame in gruppi per valore della chiave e applica a ciascun gruppo una o più funzioni di aggregazione | `df.groupby("Category").agg(Fatturato=("SalesAmount", "sum"))` |
| `merge()` e `concat()` | `merge()` affianca le colonne di due DataFrame in base a una chiave comune; `concat()` accoda righe o colonne | `pd.merge(sales, products, on="ProductID", how="left")` |
| Workflow Scikit-learn | sequenza comune a ogni estimator: split dei dati, creazione del modello, fit() sul training, predict() e valutazione sul test | `model = LinearRegression().fit(X_train, y_train)`; `model.score(X_test, y_test)` |

# Task 1 · Python di base sui dati forniti
*Python for Data Science · Parti 1 e 2 · Giorno 1 · circa 1,5 ore*

**Consegna:** sui quattro file forniti dal docente, svolgere i primi calcoli con strutture dati, cicli e funzioni Python, senza metodi di analisi Pandas.

1. leggere `products.csv` e `sales.json` e convertirli con `.to_dict(orient="records")` in liste di dizionari; costruire `regions` come dizionario `{RegionID: (State, SalesRegion)}` e il set delle categorie.
2. con una list comprehension e un costrutto `if/elif/else` assegnare a ogni prodotto la fascia di prezzo: Bassa sotto 20 €, Media fino a 60 €, Alta oltre.
3. scrivere la funzione `fatturato_categoria(vendite, prodotti, categoria)` che con un ciclo `for` somma `SalesAmount` della categoria, salta i valori mancanti e restituisce il totale con `return`.
4. con un ciclo `while` contare le transazioni necessarie a superare 10.000 € di fatturato cumulato; mostrare le prime cinque transazioni con `vendite[:5]`.

**Vincolo:** Pandas si usa solo per leggere i file e convertirli; somme, conteggi e classificazioni si calcolano con costrutti Python.

**Definizione richiesta:** differenza tra lista, tupla, set e dizionario per mutabilità, ordine, duplicati e modalità di accesso (indice o chiave).

> **Criterio di completamento:** `len()` di lista prodotti, dizionario regions e set delle categorie riportata; fascia assegnata a ogni prodotto; fatturato per categoria da `fatturato_categoria()`, da confrontare con il Task 3b.

In [1]:
# Es 1 Punto 1

import pandas as pd

# dataframe
products_df = pd.read_csv("data/products.csv")
sales_df = pd.read_json("data/sales.json")
regions_df = pd.read_excel("data/regions.xlsx")
resellers_df = pd.read_csv("data/resellers.csv")

# liste di dizionari
products = products_df.to_dict(orient="records")
sales = sales_df.to_dict(orient="records")

# dizionario regions
regions = {
    r["RegionID"]: (r["State"], r["SalesRegion"])
    for r in regions_df.to_dict(orient="records")
}

# Set delle categorie
categories = {p["Category"] for p in products}

print("Products:", len(products))
print("Sales:", len(sales))
print("Regions:", len(regions))
print("Categories:", len(categories), categories)

Products: 12
Sales: 2000
Regions: 9
Categories: 3 {'Giocattoli', 'Outdoor', 'Giochi da tavolo'}


In [2]:
# Es 1 punto 2

def price_band(price):
    if price < 20:
        return "Bassa"
    elif price <= 60:
        return "Media"
    else:
        return "Alta"

bands = [price_band(p["ListPrice"]) for p in products]

# nuova lista di dizionari con la fascia di prezzo, senza modificare products

products_banded = [{**p, "PriceBand": price_band(p["ListPrice"])} for p in products]

# verifiche
print([p for p in products if p["ListPrice"] != p["ListPrice"]])
products_banded[:5]


[]


[{'ProductID': 1,
  'ProductName': 'Bici Junior',
  'Category': 'Outdoor',
  'ListPrice': 129.0,
  'PriceBand': 'Alta'},
 {'ProductID': 2,
  'ProductName': 'Bici BMX',
  'Category': 'Outdoor',
  'ListPrice': 189.0,
  'PriceBand': 'Alta'},
 {'ProductID': 3,
  'ProductName': 'Monopattino',
  'Category': 'Outdoor',
  'ListPrice': 79.0,
  'PriceBand': 'Alta'},
 {'ProductID': 4,
  'ProductName': 'Pallone Calcio',
  'Category': 'Outdoor',
  'ListPrice': 19.9,
  'PriceBand': 'Bassa'},
 {'ProductID': 5,
  'ProductName': 'Racchette Beach',
  'Category': 'Outdoor',
  'ListPrice': 14.5,
  'PriceBand': 'Bassa'}]

In [3]:
# Es 1 punto 3

def fatturato_categoria(sales, products, category):
    # 1. set dei ProductID che appartengono alla categoria
    product_ids = {p["ProductID"] for p in products if p["Category"] == category}

    # 2. totale iniziale
    total = 0

    # 3. ciclo for sulle vendite
    for s in sales:
        amount = s["SalesAmount"]
        if amount != amount:                 # nan: valore mancante, si salta
            continue
        if s["ProductID"] in product_ids:    # vendita della categoria richiesta
            total += amount

    # 4. restituzione del totale
    return total


# fatturato per ogni categoria
for c in categories:
    print(c, round(fatturato_categoria(sales, products, c), 2))

Giocattoli 174832.0
Outdoor 331998.4
Giochi da tavolo 53245.1


### Task 1 · Punto 3
Funzione `fatturato_categoria()`: con un ciclo `for` somma `SalesAmount` delle vendite della categoria, saltando i valori mancanti, e restituisce il totale con `return`.

Fatturato per categoria:
- Giochi da tavolo: 53.245,10 €
- Giocattoli: 174.832,00 €
- Outdoor: 331.998,40 €

In [4]:
# Es 1 punto 4

total = 0
count = 0

while total <= 10000 and count < len(sales):
    amount = sales[count]["SalesAmount"]
    if amount == amount:          # salta i nan
        total += amount
    count += 1

print("Transazioni necessarie:", count)
print("Fatturato cumulato:", round(total, 2))

# prime cinque transazioni
sales[:5]

Transazioni necessarie: 56
Fatturato cumulato: 10164.8


[{'SalesID': 1194,
  'ProductID': 6,
  'RegionID': 7,
  'ResellerID': 283,
  'SalesDate': '2023-01-01',
  'Quantity': 4.0,
  'SalesAmount': 71.6},
 {'SalesID': 859,
  'ProductID': 6,
  'RegionID': 8,
  'ResellerID': 28,
  'SalesDate': '2023-01-02',
  'Quantity': 7.0,
  'SalesAmount': nan},
 {'SalesID': 1747,
  'ProductID': 12,
  'RegionID': 3,
  'ResellerID': 281,
  'SalesDate': '2023-01-02',
  'Quantity': 2.0,
  'SalesAmount': 118.0},
 {'SalesID': 279,
  'ProductID': 3,
  'RegionID': 7,
  'ResellerID': 128,
  'SalesDate': '2023-01-02',
  'Quantity': 3.0,
  'SalesAmount': nan},
 {'SalesID': 1470,
  'ProductID': 1,
  'RegionID': 6,
  'ResellerID': 137,
  'SalesDate': '2023-01-04',
  'Quantity': 4.0,
  'SalesAmount': nan}]

### Task 1 · Definizione richiesta
**Differenza tra lista, tupla, set e dizionario**

| Struttura | Sintassi | Mutabile | Ordinata | Duplicati | Accesso |
|---|---|---|---|---|---|
| **Lista** | `[1, 2, 3]` | Sì | Sì | Ammessi | Per indice: `lista[0]` |
| **Tupla** | `(1, 2, 3)` | No | Sì | Ammessi | Per indice: `tupla[0]` |
| **Set** | `{1, 2, 3}` | Sì | No | Non ammessi | Nessun indice: solo appartenenza (`x in set`) o iterazione |
| **Dizionario** | `{"a": 1}` | Sì | Sì, ordine di inserimento (da Python 3.7) | Chiavi uniche, valori ripetibili | Per chiave: `diz["a"]` |

- **Mutabilità:** lista, set e dizionario si possono modificare dopo la creazione (aggiungere, togliere, cambiare elementi); la tupla no, una volta creata resta fissa.
- **Ordine:** lista e tupla mantengono la posizione degli elementi; il dizionario mantiene l'ordine di inserimento delle chiavi; il set non garantisce alcun ordine.
- **Duplicati:** il set li elimina automaticamente; nel dizionario le chiavi sono uniche.
- **Accesso:** lista e tupla per indice numerico, dizionario per chiave, set solo verificando se un elemento è presente.

**Nel Task 1:**
- `products` e `sales` sono **liste** di dizionari: sequenze ordinate di record da scorrere con un ciclo.
- Ogni singolo prodotto o vendita è un **dizionario**: si accede ai campi per chiave (`p["ListPrice"]`).
- `regions` è un **dizionario** `{RegionID: (State, SalesRegion)}`: tabella di consultazione con accesso diretto per chiave.
- La coppia `(State, SalesRegion)` è una **tupla**: due valori fissi che non devono cambiare.
- `categories` è un **set**: raccoglie le categorie una sola volta, eliminando i duplicati.